X_test = [
  [1.5, 2.5]
]
X_train = [
  [1, 2],
  [2, 3],
  [3, 4],
  [4, 5]
]
y_train = [
  0,
  0,
  1,
  1
]

In [ ]:
import numpy as np


class GradientBoostingClassifier:
    def __init__(self, n_estimators: int = 10, learning_rate: float = 0.1):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.bias = 0.0
        self.stumps = []

    def _sigmoid(self, z):
        return 1.0 / (1.0 + np.exp(-z))

    def _fit_stump(self, X, r):
        n_samples, n_features = X.shape
        best = None
        best_sse = np.inf
        for j in range(n_features):
            values = X[:, j]
            uniq = np.unique(values)
            if uniq.size < 2:
                continue
            thresh = (uniq[:-1] + uniq[1:]) / 2.0
            for t in thresh:
                left = values <= t
                right = ~left
                if left.sum() == 0 or right.sum() == 0:
                    continue
                l_mean = r[left].mean()
                r_mean = r[right].mean()
                pred = np.where(left, l_mean, r_mean)
                sse = np.sum((r - pred) ** 2)
                if sse < best_sse:
                    best_sse = sse
                    best = (j, t, l_mean, r_mean)
        if best is None:
            const = r.mean() if n_samples > 0 else 0.0
            return (-1, 0.0, const, const)
        return best

    def _stump_predict(self, X, stump):
        j, t, l_mean, r_mean = stump
        if j == -1:
            return np.full(X.shape[0], l_mean)
        left = X[:, j] <= t
        return np.where(left, l_mean, r_mean)

    def fit(self, X, y) -> None:
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        eps = 1e-6
        p0 = np.clip(y.mean(), eps, 1.0 - eps)
        self.bias = np.log(p0 / (1.0 - p0))
        self.stumps = []
        score = np.full(X.shape[0], self.bias)
        for _ in range(self.n_estimators):
            p = self._sigmoid(score)
            r = y - p
            stump = self._fit_stump(X, r)
            self.stumps.append(stump)
            pred = self._stump_predict(X, stump)
            score += self.learning_rate * pred

    def predict_proba(self, X):
        X = np.asarray(X, dtype=float)
        score = np.full(X.shape[0], self.bias)
        for stump in self.stumps:
            score += self.learning_rate * self._stump_predict(X, stump)
        p = self._sigmoid(score)
        return p.tolist()

# Instantiate model
model = GradientBoostingClassifier()
# Fit model
model.fit(X_train, y_train)
# Get predictions
y_test = model.predict_proba(X_test)
# Output predictions
y_test